# Lab 0 · Your first LLM call

This notebook takes about 15–20 minutes. The goal is to confirm that your machine can call an LLM through LangChain, and to see the ideas from Module 0 at work: messages, streaming, temperature, tokens and structured output.

**Before you run it:** choose the `.venv` kernel from the `agentic-ai` folder (top right of the notebook), then run one cell at a time with `Shift+Enter`.

## 1. Choose a model

`get_llm()` reads `.env` to decide which provider and model to use. Every lab in this course uses this one function, so you can switch between a local model and an API without changing code.

In [ ]:
from agentlab import get_llm, provider_settings

provider, model, _ = provider_settings()
print(f"provider = {provider}\nmodel    = {model}")

llm = get_llm()

## 2. The first call

`invoke()` sends text and waits for the whole answer. What comes back is a message carrying both the text and the number of tokens used.

In [ ]:
reply = llm.invoke("อธิบายคำว่า token ในบริบทของ LLM ภายใน 2 ประโยค")

print(reply.text)
print("\nusage:", reply.usage_metadata)

## 3. Messages and roles

In Module 0 we saw that the API takes a list of messages, each with a role. `system` sets the rules; `user` is the question. Edit the `system` text and see how the answer changes.

In [ ]:
messages = [
    ("system", "You are an accounting assistant. Answer in Thai, in one short sentence."),
    ("user", "ใบกำกับภาษีต่างจากใบเสร็จรับเงินอย่างไร"),
]
print(llm.invoke(messages).text)

## 4. Streaming

The model writes its answer one token at a time. `stream()` shows each piece as soon as it is produced, with no waiting for the end.

In [ ]:
for chunk in llm.stream("นับเลข 1 ถึง 10 เป็นภาษาไทย คั่นด้วยจุลภาค"):
    print(chunk.text, end="|", flush=True)

## 5. Temperature

Run the same prompt 3 times at temperature 0 and 3 more times at 1.2, then compare which set repeats itself more.

In [ ]:
prompt = "ตั้งชื่อร้านกาแฟ 1 ชื่อ ตอบเฉพาะชื่อร้าน"

for temperature in (0, 1.2):
    model_t = get_llm(temperature=temperature)
    names = [model_t.invoke(prompt).text.strip() for _ in range(3)]
    print(f"temperature={temperature}: {names}")

## 6. Does Thai really take more tokens?

The two sentences below mean the same thing. We look at the input token count the model reports. The count includes the message framing, which is the same in both cases, so the difference comes from the sentences themselves.

In [ ]:
sentences = {
    "en": "The customer must pay the invoice within thirty days of the delivery date.",
    "th": "ลูกค้าต้องชำระเงินตามใบแจ้งหนี้ภายในสามสิบวันนับจากวันที่ส่งมอบสินค้า",
}
for lang, sentence in sentences.items():
    usage = llm.invoke(sentence + "\n\nReply with: ok").usage_metadata
    print(f"{lang}: {len(sentence):3d} characters -> {usage['input_tokens']} input tokens")

## 7. Structured output

Document intelligence needs data a program can use. We define a schema with Pydantic and have the model answer in that shape. This is the main technique in Lab 1.

In [ ]:
from pydantic import BaseModel, Field


class Invoice(BaseModel):
    invoice_no: str = Field(description="Invoice number exactly as printed")
    seller: str
    total: float = Field(description="Grand total including VAT")
    currency: str = Field(description="ISO currency code such as THB")


text = """
ใบแจ้งหนี้ เลขที่ INV-2026-0042
ผู้ขาย: บริษัท สยามเปเปอร์ จำกัด
กระดาษ A4 80 แกรม 50 รีม @ 110.00 = 5,500.00
ภาษีมูลค่าเพิ่ม 7% = 385.00
รวมทั้งสิ้น 5,885.00 บาท
"""

invoice = llm.with_structured_output(Invoice).invoke(f"Extract the invoice fields.\n{text}")
print(invoice)
print("total is a number:", invoice.total + 1)

## 8. Switch provider

Pass a provider name to `get_llm()` to use a different one for a moment. This cell tries every provider you have set up and skips those without a key.

In [ ]:
import time

from agentlab import PROVIDERS

for name in PROVIDERS:
    try:
        start = time.time()
        answer = get_llm(name).invoke("Reply with exactly one word: ready").text.strip()
        print(f"{name:10s} {answer!r} in {time.time() - start:.1f}s")
    except Exception as error:
        print(f"{name:10s} skipped: {str(error)[:90]}")

## Try it yourself

1. Edit the `system` message in section 3 so the model answers in English as 3 bullet points.
2. Add a field `vat: float` to `Invoice` and re-run section 7. Does the model extract it correctly?
3. Change the total in the invoice text so it no longer matches the sum of the items, then ask the model whether the total is correct. Note how it answers. We return to this problem in Lab 2.

**What to hand in:** see the last step of the Lab 0 guide.